In [1]:
import os
import json
import time
import re
import pandas as pd
import concurrent.futures
import httpx
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(dotenv_path='/home/check/DATA/university/yr3, hk2/IE403 - Khai thác dữ liệu truyền thông xã hội/API.env')

True

In [2]:
TOGETHER_API_KEY = os.getenv("Together_AI_API") 

client = OpenAI(
    api_key=TOGETHER_API_KEY,
    base_url="https://api.together.xyz/v1",
    timeout=httpx.Timeout(120.0) 
)

# Chuyển đổi sang Model Flagship mạnh nhất của nhà Qwen
MODEL_JUDGE = "Qwen/Qwen3-235B-A22B-Instruct-2507-tput"

In [3]:
# ==========================================
# 2. HÀM GỌI API GÁN NHÃN (ĐÃ FIX LỖI STREAM CHUNK RỖNG)
# ==========================================
def get_sarcasm_labels_qwen(batch_data):
    system_prompt = """Bạn là một CHUYÊN GIA NGÔN NGỮ HỌC và HỆ THỐNG ĐÁNH GIÁ (LLM-as-a-Judge) cấp cao.
Nhiệm vụ của bạn là gán nhãn "Sarcastic" (Mỉa mai) hoặc "Non-Sarcastic" (Không mỉa mai) cho các bình luận. NẾU bình luận là "Sarcastic", BẮT BUỘC phân loại nhóm mỉa mai (sarcasm_type).

<sarcasm_guideline>
1. [Context] và [Comment] là sự thật tuyệt đối. [Commonsense] chỉ là tham khảo, hãy bỏ qua nếu nó vô lý.
2. Mỉa mai là sự mâu thuẫn (Incongruity) giữa ngữ cảnh thực tế và ý nghĩa bề mặt của bình luận (Lời khen giả tạo, Vi phạm kỳ vọng).
3. ĐẶC TRƯNG TIẾNG VIỆT: Chú ý "Nói mát / Lịch sự giả tạo", "Khen ngợi lạc đề", và các từ tình thái ("nhỉ", "ghê", "ha", "cơ") đặt trong hoàn cảnh tồi tệ.

PHÂN LOẠI 5 NHÁNH MỈA MAI (SARCASM TYPES):
1. "Lexical Contradiction": Mâu thuẫn từ vựng (Lời khen giả tạo, từ tình thái).
2. "Propositional Contradiction": Mâu thuẫn mệnh đề (Lịch sự giả tạo, Khen lạc đề).
3. "Rhetorical Question": Câu hỏi tu từ thách thức, phủ định.
4. "Wh-Question": Câu hỏi móc mỉa dùng Ai, Cái gì, Ở đâu, Sao.
5. "Hypothetical": Giả định phóng đại, phi logic.

REASONING CHAIN:
- Khen/đồng tình + Thực tế tồi tệ => Sarcastic (Lexical / Propositional).
- Câu hỏi thách thức => Sarcastic (Rhetorical / Wh-Question).
- Giả định phi lý => Sarcastic (Hypothetical).
- Chửi bới, chê bai trực tiếp hoặc khen đúng sự thật => Non-Sarcastic.
</sarcasm_guideline>

<output_format>
BẠN BẮT BUỘC TRẢ VỀ JSON CHỨA MẢNG "results". MỖI PHẦN TỬ GỒM:
- "id": Trích xuất từ Input.
- "sarcasm_label": "Sarcastic" HOẶC "Non-Sarcastic".
- "sarcasm_type": 1 trong 5 loại trên. NẾU "Non-Sarcastic", điền "None".
- "reasoning": Giải thích ngắn (DƯỚI 40 CHỮ) về sự mâu thuẫn.
</output_format>

[Input Batch]
(Dữ liệu của bạn ở đây)
"""

    user_prompt = "[Input Batch]\n"
    for item in batch_data:
        user_prompt += f"ID: {item['id']} | Context: {item['ctx']} | Comment: {item['cmt']} | Commonsense: {item['cs']}\n"
    user_prompt += "\n[Output (JSON)]:"

    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL_JUDGE,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.0,
                response_format={"type": "json_object"},
                max_tokens=4096,
                stream=False
            )
            
            content = response.choices[0].message.content or ""
                        
            if not content: 
                tqdm.write(f"⚠️ Batch rỗng nội dung (Lần {attempt+1}/3)")
                time.sleep(2)
                continue
                
            try:
                # Xóa các ký tự ẩn gây lỗi JSON
                content = re.sub(r'[\x00-\x1f]', '', content)
                result_json = json.loads(content, strict=False)
                
                if isinstance(result_json, list):
                    return result_json
                # Nếu Qwen trả về Dict (có bọc "results") -> Dùng .get()
                elif isinstance(result_json, dict):
                    return result_json.get("results", [])
                else:
                    return None
                    
            except json.JSONDecodeError as e:
                tqdm.write(f"⚠️ Lỗi parse JSON (Lần {attempt+1}/3): {e}. Đầu output: {content[:300]}")
                time.sleep(2)
                continue

        except Exception as e:
            tqdm.write(f"⚠️ Lỗi API Qwen Max (Lần {attempt+1}/3): {str(e)}")
            time.sleep(4) 
            
    return None

In [4]:
def process_single_batch(batch_info):
    batch_idx, batch_data = batch_info
    results = get_sarcasm_labels_qwen(batch_data)
    return batch_idx, results

In [5]:
def process_dataset_labeling(input_csv, output_csv, batch_size=15, max_workers=3):
    if os.path.exists(output_csv):
        print(f"\n[*] Tìm thấy file đang chạy dở: {output_csv}. Đang resume...")
        df = pd.read_csv(output_csv)
    else:
        print(f"\n[*] Chạy lần đầu. Đang đọc file gốc: {input_csv}")
        df = pd.read_csv(input_csv)
        df['sarcasm_label'] = None 
        df['sarcasm_type'] = None
        df['reasoning'] = None
        df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    
    if 'sarcasm_label' not in df.columns: df['sarcasm_label'] = None
    if 'sarcasm_type' not in df.columns: df['sarcasm_type'] = None
    if 'reasoning' not in df.columns: df['reasoning'] = None

    def needs_processing(x):
        return str(x).strip() in ['nan', 'None', '']

    pending_indices = df[df['sarcasm_label'].apply(needs_processing)].index.tolist()
    print(f"Tổng số dòng chờ Qwen Max gán nhãn: {len(pending_indices)} | Batch: {batch_size} | Luồng: {max_workers}")
    
    if not pending_indices:
        print("🎉 Mọi dữ liệu đã được gán nhãn hoàn tất 100%!")
        return

    # Chia mảng dữ liệu thành các cục (Batches)
    all_batches = []
    for i in range(0, len(pending_indices), batch_size):
        batch_indices = pending_indices[i:i+batch_size]
        batch_data = []
        for idx in batch_indices:
            ctx = str(df.at[idx, 'video_core_content']).replace('\n', ' ')
            cmt = str(df.at[idx, 'comment']).replace('\n', ' ')
            cs = str(df.at[idx, 'commonsense']).replace('\n', ' ')
            
            if cmt.lower() == 'nan' or cs.lower() == 'nan': 
                continue
                
            batch_data.append({"id": idx, "ctx": ctx, "cmt": cmt, "cs": cs})
        
        if batch_data:
            all_batches.append((i // batch_size, batch_data))

    print(f"🚀 Kích hoạt cỗ máy Giám khảo siêu cấp Qwen Max...")

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_single_batch, b) for b in all_batches]
        
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Đang Judge (Qwen Max)"):
            batch_idx, results = future.result()
            
            if results:
                for res in results:
                    try:
                        row_id = int(res.get("id"))
                    except (TypeError, ValueError):
                        tqdm.write(f"⚠️ Bỏ qua kết quả có id không hợp lệ: {res}")
                        continue

                    label = res.get("sarcasm_label", "")
                    sarcasm_type = res.get("sarcasm_type", "")
                    reason = res.get("reasoning", "")

                    if label not in ["Sarcastic", "Non-Sarcastic"]:
                        tqdm.write(f"⚠️ Bỏ qua nhãn không hợp lệ tại dòng {row_id}: {label}")
                        continue
                        
                    if row_id is not None and row_id in df.index:
                        df.at[row_id, 'sarcasm_label'] = str(label).strip()
                        df.at[row_id, 'sarcasm_type'] = str(sarcasm_type).strip()
                        df.at[row_id, 'reasoning'] = str(reason).strip()
            
            # Lưu đè liên tục để bảo vệ dữ liệu
            df.to_csv(output_csv, index=False, encoding='utf-8-sig')

    print(f"\n[HOÀN THÀNH TẬP GOLD DATA] Toàn bộ dữ liệu nhãn chuẩn đã lưu tại: {output_csv}")

In [6]:
if __name__ == "__main__":
    INPUT_FILE = "/home/check/DATA/university/yr3, hk2/IE403 - Khai thác dữ liệu truyền thông xã hội/Project/data/gold/test_data_infer.csv" 
    OUTPUT_FILE = "Qwen3_A22B_zero-shot.csv" 
    
    BATCH_SIZE = 10 
    MAX_WORKERS = 5
    
    if os.path.exists(INPUT_FILE):
        process_dataset_labeling(INPUT_FILE, OUTPUT_FILE, batch_size=BATCH_SIZE, max_workers=MAX_WORKERS)
    else:
        print(f"Lỗi: Không tìm thấy file gốc {INPUT_FILE}")


[*] Tìm thấy file đang chạy dở: Qwen3_A22B_zero-shot.csv. Đang resume...
Tổng số dòng chờ Qwen Max gán nhãn: 1 | Batch: 10 | Luồng: 5
🚀 Kích hoạt cỗ máy Giám khảo siêu cấp Qwen Max...


Đang Judge (Qwen Max): 100%|██████████| 1/1 [00:30<00:00, 30.34s/it]


[HOÀN THÀNH TẬP GOLD DATA] Toàn bộ dữ liệu nhãn chuẩn đã lưu tại: Qwen3_A22B_zero-shot.csv
